# 02 — one-to-one linkage and descriptive statistics

this notebook covers descriptive work for nacc path c. hypothesis tests are not part of phase 1.

## 1. opening the files

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
                                                                                
ROOT = Path.cwd()
if not (ROOT / 'README.md').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw'
assert DATA.exists(), 'Place the three original CSVs in data/raw/; read data/README.md'
u = pd.read_csv(DATA / 'udslong_imputed.csv', low_memory=False)
m = pd.read_csv(DATA / 'mriscanclean.csv', low_memory=False)
print('UDS:', u.shape, 'MRI:', m.shape)


## 2. matching participants
i match people by naccid rather than by row order. this avoids mixing records if the file order changes. i also check that diagnosis codes agree.

In [ ]:
assert u.NACCID.is_unique and m.NACCID.is_unique
print('UDS-only:',len(set(u.NACCID)-set(m.NACCID)))
print('MRI-only:',len(set(m.NACCID)-set(u.NACCID)))
diagnostic_agreement = u.set_index('NACCID').NACCUDSD.sort_index().eq(m.set_index('NACCID').NACCUDSD.sort_index()).all()
print('Diagnosis agreement:',diagnostic_agreement)
assert diagnostic_agreement
d = u.merge(m.drop(columns='NACCUDSD'),on='NACCID',validate='one_to_one')
print('Final participant rows:',len(d))

## 3. combining left and right measures
i add left and right volumes because they describe different regions. for thickness i average the two sides. i mark a nonpositive volume as missing instead of letting a false zero lower the sum.

In [ ]:
left_v=d.LH_ENTORHINAL_GVOL.where(d.LH_ENTORHINAL_GVOL>0)
right_v=d.RH_ENTORHINAL_GVOL.where(d.RH_ENTORHINAL_GVOL>0)
d['entorhinal_volume']=left_v+right_v
d['entorhinal_thickness']=(d.LH_ENTORHINAL_AVGTH+d.RH_ENTORHINAL_AVGTH)/2
print(d[['CDRSUM','entorhinal_volume','entorhinal_thickness']].describe())

## 4. comparing groups
the group numbers are labels rather than measurements. i use counts for diagnosis and numerical summaries for the actual scores and brain measures.

In [ ]:
print(d.NACCUDSD.value_counts().sort_index())
print(d.groupby('NACCUDSD').agg(n=('CDRSUM','size'),median_cdr=('CDRSUM','median'),
                                volume_mean=('entorhinal_volume','mean'),
                                thickness_mean=('entorhinal_thickness','mean')).round(2))

## 5. making the plots
the script creates nine figures. histograms show the distributions and boxplots make group differences easier to see. scatterplots are shown without a fitted model.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'src'/'phase1_analysis.py')],check=True)
from IPython.display import Image,display
display(Image(filename=str(ROOT/'figures'/'fig05_thickness_by_diagnosis.png'),width=700))

the original uds-only participants are not in these files. we cannot compare scanned and unscanned people without that missing group.